In [ ]:
#| echo: false
import os

from IPython.display import HTML

import talk

# Rendered by Quarto, slides have no Python kernel behind them, so every widget
# is baked in as a self-contained static page (render_static) instead of a live
# anywidget, with its settings sidebar closed to leave the slide to the
# chart. Run in Jupyter, the widgets stay live.
if "QUARTO_DOCUMENT_PATH" in os.environ:
    from retentioneering.widgets._base import RetentioneeringWidget

    # A widget can ask for its own height on a slide: `widget.slide_height = …`
    # (the widget's layout follows its `height`, the iframe follows the page).
    SLIDE_WIDGET_HEIGHT = 470

    def _static_bundle(self, include=None, exclude=None, **kwargs):
        height = getattr(self, "slide_height", None)
        if height:
            self.height = height
        return {"text/html": self.render_static(height=height or SLIDE_WIDGET_HEIGHT, sidebar_open=False).data}

    RetentioneeringWidget._repr_mimebundle_ = _static_bundle

# The whole case, run once: the slides show the calls and read the results.
stream = talk.load()
A, B, C = talk.A, talk.B, talk.C
dropoffs = talk.select_dropoffs(stream)
MAY_11 = talk.mature_before(stream)
cropped = talk.crop(dropoffs, MAY_11)
clustered = talk.cluster(cropped)
payment_sessions = stream.filter_paths({"has_event": B}, path_col="session_id")

## The funnel shows where, not why

<p class="lead-question">2,259 people reached the payment screen. 280 of them, 12%, never ordered. Why?</p>

In [ ]:
#| echo: false
funnel = stream.funnel(steps=[A, B, C], path_col="user_id")
funnel.slide_height = 420
funnel

- 1,979 of them ordered: the last step loses 12%
- A funnel counts the people who left. It can't show what they did instead

## Select: reached payment, never ordered

<p class="idea-pattern"><code>checkout_start->.&#42;->checkout_payment->[^order_success]&#42;->path_end</code></p>
<p class="idea-lead">Keep only the failed paths (<span class="failed-mark">✗</span>): a checkout start, later the payment screen, and no order from there to the end. <b>280 people</b></p>

In [ ]:
#| echo: false
HTML(talk.pattern_figure())

## Cut a window after the payment screen

<p class="idea-pattern"><span>from</span><code>checkout_start->.&#42;->checkout_payment</code><span>to</span><code>the same + 10 events</code></p>
<p class="idea-lead">Keep the payment screen and the 10 events after it: what happened next. Only people who had two weeks to come back: <b>212 people</b></p>

In [ ]:
#| echo: false
HTML(talk.window_figure(clustered, dropoffs))

In [ ]:
#| echo: false
before = talk.whole_paths_graph(dropoffs)
before.slide_height = 270
before

In [ ]:
#| echo: false
after = talk.route_graph(clustered)
after.slide_height = 270
after

## Clustering: six ways not to order

In [ ]:
#| echo: false
overview = talk.ordered_columns(clustered.segment_overview("dropoff", metrics=talk.OVERVIEW, path_col="user_id"))
overview.slide_height = 312
overview

In [ ]:
#| echo: false
graph = talk.route_graph(clustered)
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "payment · gone")
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "review · gone")
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "error · gone")
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "payment · back")
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "review · back")
graph.slide_height = 300
graph

In [ ]:
#| echo: false
graph = talk.route_graph(clustered, "error · back")
graph.slide_height = 300
graph

- Cluster the windows by what is in them: a payment error, an order review, the cart, product pages, a return in a new session
- Two questions name the six clusters. What happened at the payment screen: a payment `error`, an order `review`, or nothing after the `payment` screen. Did they come `back` in a new session, or are they `gone`
- `length`: the payment screen and up to 10 events. The `gone` windows are 1–3 events long

## The error clusters point at one build

In [ ]:
#| echo: false
builds = talk.ordered_columns(clustered.segment_overview(
    "dropoff",
    metrics=talk.OVERVIEW
    + [
        {
            "metric": "in_segment_bulk",
            "metric_args": {"segment_name": "build", "segment_levels": talk.ANDROID_BUILDS},
            "agg": "mean",
        }
    ],
    path_col="user_id",
))
builds.slide_height = 430
builds

- Add the app build to the same table: the share of people in each cluster on each Android build
- The two `error` clusters hold three times as many **android 4.12.0** users as the others
- Across all 2,426 payment sessions, a payment error comes in 37% of them on android 4.12.0, 6–12% on any other build

## Step matrix: android 4.12.0 against the rest

In [ ]:
#| echo: false
on_android = {"metric": "in_segment", "metric_args": {"segment_name": "build", "segment_level": talk.ANDROID}, "op": "=", "value": True}
android = payment_sessions.filter_paths(on_android, path_col="session_id").step_matrix(anchor=B, max_steps=3, path_col="session_id")
android.slide_height = 420
android

In [ ]:
#| echo: false
rest = payment_sessions.filter_paths({"op": "not", "args": [on_android]}, path_col="session_id").step_matrix(anchor=B, max_steps=3, path_col="session_id")
rest.slide_height = 420
rest

In [ ]:
#| echo: false
diff = payment_sessions.step_matrix(anchor=B, max_steps=3, diff=("build", talk.ANDROID, "<REST>"), path_col="session_id")
diff.slide_height = 420
diff

- Every session with a payment screen, centered on it: **android 4.12.0**, **the rest**, and one minus the other
- One step later: a payment error in 37% of sessions against 8%. A bug ticket for one build

## What happened to the payment drop-offs

In [ ]:
#| echo: false
HTML(talk.shares_figure(clustered))

- Stopped at the payment screen or after the order review: questions for UX research
- A payment error, three times as often on **android 4.12.0**: a bug ticket
- Half came back later, and still didn't order

## What we called

::: {.columns}
::: {.column width="50%"}
### Data processors

- `filter_paths` with a path pattern
- `truncate_paths` with anchors
- `add_clusters`
:::

::: {.column width="50%"}
### Widgets

- `cluster_analysis`
- `transition_graph`
- `step_matrix` in diff mode
:::
:::

> The strength of Retentioneering is in combining them: data processors shape the paths, widgets show what is in them